In [2]:
!pip install -q altair pandas vl-convert-python

import pandas as pd
import numpy as np
import altair as alt
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from google.colab import data_table
import os

data_table.enable_dataframe_formatter()

archivo_mundo = "Mundo - Evolución Normativa y Restricciones 2.csv"
df_mundo = pd.read_csv(archivo_mundo)

for col in df_mundo.columns:
    if df_mundo[col].dtype == "object":
        df_mundo[col] = df_mundo[col].astype(str).str.strip()

categorias_animales = df_mundo["Animales Afectados"].dropna().unique().tolist()
paleta_solida = ["#E63946", "#1D3557", "#FB8500", "#2A9D8F", "#8338EC", "#9C6644", "#F15BB5", "#3A86FF", "#FFB703", "#06D6A0", "#D00000", "#3F37C9", "#432818", "#7209B7", "#E07A5F"]

color_map_animales = {}
for i, cat in enumerate(categorias_animales):
    color_map_animales[cat] = paleta_solida[i % len(paleta_solida)]

color_map_heatmap_mundo = {"Prohibición nacional": "#386641", "Prohibición progresiva": "#bc4749", "Aprobación de ley": "#a7c957", "Vigente": "#457b9d", "En implementación": "#f4a261"}

df_pie_mundo = df_mundo["Animales Afectados"].value_counts().reset_index()
df_pie_mundo.columns = ["Tipo", "Cantidad"]
df_pie_mundo["Porcentaje"] = (df_pie_mundo["Cantidad"] / df_pie_mundo["Cantidad"].sum()) * 100
df_pie_mundo["Etiqueta_Completa"] = df_pie_mundo["Tipo"] + " (" + df_pie_mundo["Porcentaje"].round(1).astype(str) + "%)"
df_pie_mundo = df_pie_mundo.sort_values("Tipo").reset_index(drop=True)
df_pie_mundo["Acumulado"] = df_pie_mundo["Porcentaje"].cumsum()
df_pie_mundo["Mid_Percent"] = df_pie_mundo["Acumulado"] - (df_pie_mundo["Porcentaje"] / 2)
df_pie_mundo["Angulo_Label"] = (df_pie_mundo["Mid_Percent"] / 100) * 360

pie_mundo = alt.Chart(df_pie_mundo).mark_arc(innerRadius=90, outerRadius=160, stroke="white", strokeWidth=2).encode(
    theta=alt.Theta("Cantidad:Q", stack=True),
    color=alt.Color("Tipo:N", scale=alt.Scale(domain=list(color_map_animales.keys()), range=list(color_map_animales.values())), legend=None),
    order=alt.Order("Tipo:N", sort="ascending"),
    tooltip=["Tipo", "Cantidad", alt.Tooltip("Porcentaje:Q", format=".1f")]
)

guias_mundo = alt.Chart(df_pie_mundo).mark_arc(innerRadius=165, outerRadius=195, stroke="gray", strokeWidth=1.5).encode(
    theta=alt.Theta("Angulo_Label:Q", scale=alt.Scale(domain=[0, 360])),
    theta2="Angulo_Label:Q",
    order=alt.Order("Tipo:N", sort="ascending")
)

texto_mundo = alt.Chart(df_pie_mundo).mark_text(size=12, fontWeight="bold", align="center", baseline="middle").encode(
    theta=alt.Theta("Angulo_Label:Q", scale=alt.Scale(domain=[0, 360])),
    radius=alt.value(215),
    text="Etiqueta_Completa:N",
    color=alt.value("black"),
    order=alt.Order("Tipo:N", sort="ascending")
)

dona_final_mundo = alt.layer(pie_mundo, guias_mundo, texto_mundo).resolve_scale(theta="independent")

df_paises_mundo = df_mundo[["País", "Animales Afectados"]].copy()
df_paises_mundo["Color"] = df_paises_mundo["Animales Afectados"].map(color_map_animales).fillna("#cccccc")

leyenda_paises_mundo = alt.Chart(df_paises_mundo).mark_square(size=100, stroke="black", strokeWidth=0.5).encode(
    y=alt.Y("País:N", sort=alt.EncodingSortField(field="Animales Afectados", order="ascending"), title="Países y su Categoría"),
    color=alt.Color("Color:N", scale=None)
)

vis_torta_mundo = alt.hconcat(
    dona_final_mundo.properties(title="Tipos de animales protegidos por normativas de los siguientes países", width=450, height=450),
    leyenda_paises_mundo.properties(width=100)
).resolve_scale(color="independent")

heat_rows_mundo = []
for _, row in df_mundo.iterrows():
    heat_rows_mundo.append([row["País"], "Tipo de Restricción", row["Tipo de Restricción"]])
    heat_rows_mundo.append([row["País"], "Estatus Legal", row["Estatus Legal"]])

df_heat_mundo = pd.DataFrame(heat_rows_mundo, columns=["País", "Categoría", "Valor"])
df_heat_mundo["Color"] = df_heat_mundo["Valor"].map(color_map_heatmap_mundo).fillna("#d3d3d3")

heatmap_mundo_chart = alt.Chart(df_heat_mundo).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("Categoría:N", sort=["Tipo de Restricción", "Estatus Legal"]),
    y=alt.Y("País:N", sort=alt.EncodingSortField(field="País", order="ascending")),
    color=alt.Color("Color:N", scale=None),
    tooltip=["País", "Categoría", "Valor"]
).properties(width=300, height=600)

df_leyenda_heat_mundo = pd.DataFrame(list(color_map_heatmap_mundo.items()), columns=["Valor", "Color"])
df_leyenda_heat_mundo["Grupo"] = ["Tipo de Restricción"]*3 + ["Estatus Legal"]*2

leyenda_heatmap_mundo = alt.Chart(df_leyenda_heat_mundo).mark_square(size=200, stroke="black", strokeWidth=1).encode(
    y=alt.Y("Valor:N", sort=None, title="Simbología Heatmap"),
    color=alt.Color("Color:N", scale=None)
).properties(width=150)

vis_heatmap_final_mundo = alt.hconcat(leyenda_heatmap_mundo, heatmap_mundo_chart).resolve_scale(color="independent").properties(title="Normativas (restricciones) de los siguientes países y su respectivo estatus legal")

display(df_mundo)
display(vis_torta_mundo)
display(vis_heatmap_final_mundo)

vis_torta_mundo.save("pie_animales_afectados.html")
vis_torta_mundo.save("pie_animales_afectados.png")
with Image.open("pie_animales_afectados.png") as img:
    if img.mode in ("RGBA", "P"): img = img.convert("RGB")
    img.save("pie_animales_afectados.jpg", "JPEG", quality=95)
os.remove("pie_animales_afectados.png")

vis_heatmap_final_mundo.save("heatmap_normativas.html")
vis_heatmap_final_mundo.save("heatmap_normativas.png")
with Image.open("heatmap_normativas.png") as img:
    if img.mode in ("RGBA", "P"): img = img.convert("RGB")
    img.save("heatmap_normativas.jpg", "JPEG", quality=95)
os.remove("heatmap_normativas.png")

archivo_ind = "Industria - Adaptación y Nuevos Modelos 2.csv"
df_ind = pd.read_csv(archivo_ind)

for col in df_ind.columns:
    if df_ind[col].dtype == "object":
        df_ind[col] = df_ind[col].astype(str).str.strip()

df_ind["Uso_de_Animales"] = df_ind["Uso_de_Animales"].astype(str).str.strip()
df_ind["Modelo_de_Espectáculo"] = df_ind["Modelo_de_Espectáculo"].astype(str).str.split("/").str[0].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()
df_ind = df_ind.sort_values("Año_Fundación").reset_index(drop=True)

color_map_ind = {"Sí": "#800080", "No": "#2ca02c", "Parcial": "#ffd700", "Tradicional": "#d62728", "Contemporáneo": "#1f77b4"}

heat_rows_ind = []
for _, row in df_ind.iterrows():
    heat_rows_ind.append([row["Compañía"], "Uso de Animales", row["Uso_de_Animales"]])
    heat_rows_ind.append([row["Compañía"], "Modelo de Espectáculo", row["Modelo_de_Espectáculo"]])

df_heat_ind = pd.DataFrame(heat_rows_ind, columns=["Compañía", "Categoría", "Valor"])
df_heat_ind["Color"] = df_heat_ind["Valor"].map(color_map_ind).fillna("#d3d3d3")

vis_heatmap_ind = alt.Chart(df_heat_ind).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("Categoría:N", sort=["Uso de Animales", "Modelo de Espectáculo"]),
    y=alt.Y("Compañía:N", sort=None),
    color=alt.Color("Color:N", scale=None, legend=None),
    tooltip=["Compañía", "Categoría", "Valor"]
).properties(width=300, height=350)

df_leyenda_ind = pd.DataFrame([
    ["Uso de Animales", "Sí", "#800080"],
    ["Uso de Animales", "No", "#2ca02c"],
    ["Uso de Animales", "Parcial", "#ffd700"],
    ["Modelo", "Tradicional", "#d62728"],
    ["Modelo", "Contemporáneo", "#1f77b4"]
], columns=["Grupo", "Etiqueta", "Color"])

vis_leyenda_ind = alt.Chart(df_leyenda_ind).mark_square(size=250, stroke="black", strokeWidth=1).encode(
    y=alt.Y("Etiqueta:N", title="Simbología"),
    color=alt.Color("Color:N", scale=None, legend=None)
).properties(width=180, height=250)

vis_heatmap_final_ind = alt.hconcat(vis_leyenda_ind, vis_heatmap_ind).properties(title="Circos internacionales adaptados y nuevos modelos circenses")

df_timeline = df_ind.sort_values("Año_Fundación").copy()
df_labels = df_timeline.copy()
df_labels["Diferencia_Previa"] = df_labels["Año_Fundación"].diff().fillna(999)
df_labels["Diferencia_Siguiente"] = df_labels["Año_Fundación"].diff(-1).abs().fillna(999)
df_labels = df_labels[(df_labels["Diferencia_Previa"] <= 3) | (df_labels["Diferencia_Siguiente"] <= 3)]

texto_conflictivos = alt.Chart(df_labels).mark_text(dx=10, dy=-10, fontSize=9, color="black").encode(
    x="Año_Fundación:Q",
    y=alt.Y("Compañía:N", sort=alt.EncodingSortField(field="Año_Fundación", order="descending")),
    text="Año_Fundación:N"
)

anios_existentes = sorted([int(x) for x in df_timeline["Año_Fundación"].unique()])

linea = alt.Chart(df_timeline).mark_line(color="#87CEEB", strokeWidth=2).encode(
    x=alt.X("Año_Fundación:Q", title="Año de Fundación", axis=alt.Axis(values=anios_existentes, labelAngle=0, labelOverlap=True), scale=alt.Scale(domain=[min(anios_existentes), max(anios_existentes)])),
    y=alt.Y("Compañía:N", sort=alt.EncodingSortField(field="Año_Fundación", order="descending"))
)

puntos = alt.Chart(df_timeline).mark_circle(size=120, color="#87CEEB").encode(
    x=alt.X("Año_Fundación:Q", axis=alt.Axis(values=anios_existentes, labelOverlap=True)),
    y=alt.Y("Compañía:N", sort=alt.EncodingSortField(field="Año_Fundación", order="descending")),
    tooltip=["Compañía", "Año_Fundación", "Origen", "Uso_de_Animales", "Modelo_de_Espectáculo"]
)

vis_timeline = (linea + puntos + texto_conflictivos).properties(title="Línea de Tiempo de Compañías Circenses", width=1400, height=600)

display(df_ind)
display(vis_timeline)
display(vis_heatmap_final_ind)

vis_timeline.save("timeline_circos.html")
vis_timeline.save("timeline_circos.png")
with Image.open("timeline_circos.png") as img:
    img.convert("RGB").save("timeline_circos.jpg", "JPEG", quality=95)
os.remove("timeline_circos.png")

vis_heatmap_final_ind.save("heatmap_industria.html")
vis_heatmap_final_ind.save("heatmap_industria.png")
with Image.open("heatmap_industria.png") as img:
    img.convert("RGB").save("heatmap_industria.jpg", "JPEG", quality=95)
os.remove("heatmap_industria.png")

archivo_chi = "Chile - Tradición Familiar vs. Contemporáneo Profesionalizado 2.csv"
df_chi = pd.read_csv(archivo_chi)

for col in df_chi.columns:
    if df_chi[col].dtype == "object":
        df_chi[col] = df_chi[col].astype(str).str.strip()

df_chi["Tipo_de_Circo"] = df_chi["Tipo_de_Circo"].astype(str).str.strip()
df_chi["Tipo_de_Circo"] = df_chi["Tipo_de_Circo"].replace({"Tradicional (Diverso)": "Tradicional"})
df_chi["Formación_Predominante"] = df_chi["Formación_Predominante"].astype(str).str.split("/").str[0].str.strip()
df_chi["Estructura_Orgánica"] = df_chi["Estructura_Orgánica"].astype(str).str.split("/").str[0].str.strip()
df_chi = df_chi.sort_values(["Tipo_de_Circo", "Nombre_Agrupación"]).reset_index(drop=True)

df_pie_chi = df_chi["Tipo_de_Circo"].value_counts().reset_index()
df_pie_chi.columns = ["Tipo", "Cantidad"]
df_pie_chi["Porcentaje"] = (df_pie_chi["Cantidad"] / df_pie_chi["Cantidad"].sum()) * 100
df_pie_chi["Etiqueta"] = df_pie_chi["Porcentaje"].round(1).astype(str) + "%"
df_pie_chi["Etiqueta_Completa"] = df_pie_chi["Tipo"] + " (" + df_pie_chi["Porcentaje"].round(1).astype(str) + "%)"
df_pie_chi["Angulo_Label"] = df_pie_chi["Tipo"].map({"Tradicional": 315, "Contemporáneo": 135})

pie_chi = alt.Chart(df_pie_chi).mark_arc(innerRadius=90, outerRadius=160, stroke="white", strokeWidth=2).encode(
    theta=alt.Theta("Cantidad:Q", stack=True),
    color=alt.Color("Tipo:N", scale=alt.Scale(domain=["Tradicional", "Contemporáneo"], range=["#d62728", "#1f77b4"]), legend=alt.Legend(title="Tipo de Circo")),
    order=alt.Order("Tipo:N", sort="ascending"),
    tooltip=["Tipo", "Cantidad", alt.Tooltip("Porcentaje:Q", format=".1f")]
)

guias_chi = alt.Chart(df_pie_chi).mark_arc(innerRadius=165, outerRadius=195, stroke="gray", strokeWidth=1.5).encode(
    theta=alt.Theta("Angulo_Label:Q", scale=alt.Scale(domain=[0, 360])),
    theta2="Angulo_Label:Q"
)

texto_chi = alt.Chart(df_pie_chi).mark_text(size=13, fontWeight="bold", align="center", baseline="middle").encode(
    theta=alt.Theta("Angulo_Label:Q", scale=alt.Scale(domain=[0, 360])),
    radius=alt.value(215),
    text="Etiqueta_Completa:N",
    color=alt.value("black")
)

vis_torta_chi = alt.layer(pie_chi, guias_chi, texto_chi).resolve_scale(theta="independent").properties(title="Distribución de circos tradicionales y contemporáneos chilenos", width=550, height=450)

color_map_chi = {
    "Tradicional": "#d62728", "Contemporáneo": "#1f77b4", "Transmisión familiar": "#2ca02c", "Académica": "#ffd700",
    "Social": "#8b4513", "Universitaria": "#800080", "Talleres": "#ff7f0e", "Callejera": "#ffffff", "Autodidacta": "#808080",
    "Empresa circense": "#90ee90", "Personalidad jurídica": "#87ceeb", "Agrupación sin personalidad jurídica": "#ffb6c1",
    "Agrupación sin pers. jurídica": "#ffb6c1"
}

heat_rows_chi = []
for _, row in df_chi.iterrows():
    heat_rows_chi.append([row["Nombre_Agrupación"], "Tipo de Circo", row["Tipo_de_Circo"]])
    heat_rows_chi.append([row["Nombre_Agrupación"], "Formación Predominante", row["Formación_Predominante"]])
    heat_rows_chi.append([row["Nombre_Agrupación"], "Estructura Orgánica", row["Estructura_Orgánica"]])

df_heat_chi = pd.DataFrame(heat_rows_chi, columns=["Agrupación", "Categoría", "Valor"])
df_heat_chi["Color"] = df_heat_chi["Valor"].map(color_map_chi).fillna("#d3d3d3")

vis_heatmap_chi = alt.Chart(df_heat_chi).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("Categoría:N", sort=["Tipo de Circo", "Formación Predominante", "Estructura Orgánica"]),
    y=alt.Y("Agrupación:N", sort=None),
    color=alt.Color("Color:N", scale=None, legend=None),
    tooltip=["Agrupación", "Categoría", "Valor"]
).properties(width=450, height=600)

df_leyenda_chi = pd.DataFrame([
    ["Tipo de Circo", "Tradicional", "#d62728"], ["Tipo de Circo", "Contemporáneo", "#1f77b4"], ["Formación", "Transmisión familiar", "#2ca02c"],
    ["Formación", "Académica", "#ffd700"], ["Formación", "Social", "#8b4513"], ["Formación", "Universitaria", "#800080"],
    ["Formación", "Talleres", "#ff7f0e"], ["Formación", "Callejera", "#ffffff"], ["Formación", "Autodidacta", "#808080"],
    ["Estructura", "Empresa circense", "#90ee90"], ["Estructura", "Personalidad jurídica", "#87ceeb"], ["Estructura", "Agrupación", "#ffb6c1"]
], columns=["Grupo", "Etiqueta", "Color"])

vis_leyenda_chi = alt.Chart(df_leyenda_chi).mark_square(size=250, stroke="black", strokeWidth=1).encode(
    y=alt.Y("Etiqueta:N", title="Simbología"), color=alt.Color("Color:N", scale=None, legend=None)
).properties(width=180, height=400)

vis_heatmap_final_chi = alt.hconcat(vis_leyenda_chi, vis_heatmap_chi).properties(title="Caracterización de agrupaciones circenses chilenas")

display(df_chi)
display(vis_torta_chi)
display(vis_heatmap_final_chi)

vis_torta_chi.save("pie_circos.html")
vis_torta_chi.save("pie_circos.png")
with Image.open("pie_circos.png") as img:
    img.convert("RGB").save("pie_circos.jpg", "JPEG", quality=95)
os.remove("pie_circos.png")

vis_heatmap_final_chi.save("heatmap_circos.html")
vis_heatmap_final_chi.save("heatmap_circos.png")
with Image.open("heatmap_circos.png") as img:
    img.convert("RGB").save("heatmap_circos.jpg", "JPEG", quality=95)
os.remove("heatmap_circos.png")

,País,Año Hito,Tipo de Restricción,Animales Afectados,Estatus Legal
0,Austria,2005.0,Prohibición nacional,Animales salvajes,Vigente
1,Bolivia,2009.0,Prohibición nacional,Salvajes y domésticos,Vigente
2,Bosnia-Herzegovina,NaN,Prohibición nacional,Todos los animales,Vigente
3,Colombia,2013.0,Prohibición nacional,Animales salvajes,Vigente
4,Costa Rica,2002.0,Prohibición nacional,Animales salvajes,Vigente
5,Croacia,2006.0,Prohibición nacional,Animales salvajes,Vigente
6,Chipre,NaN,Prohibición nacional,Todos los animales,Vigente
7,Dinamarca,NaN,Prohibición nacional,Animales salvajes,Vigente
8,Ecuador,NaN,Prohibición nacional,Animales salvajes nativos,Vigente
9,El Salvador,NaN,Prohibición nacional,Animales salvajes,Vigente


alt.HConcatChart(...)

alt.HConcatChart(...)

,Compañía,Origen,Año_Fundación,Uso_de_Animales,Modelo_de_Espectáculo,Hito_Adaptación
0,Cirque Medrano,Francia,1844,No,Tradicional,Reconvertido a show humano
1,Cirque d'Hiver,Francia,1852,Parcial,Tradicional,Histórico; evolución hacia arte humano
2,Circus Krone,Alemania,1870,Sí,Tradicional,Mantiene zoológico y doma
3,Ringling Bros. and Barnum & Bailey,EE.UU.,1881,No,Tradicional,Elimina animales en 2017
4,Lennon Bros Circus,Australia,1893,No,Tradicional,Primero en Australia en dejar animales
5,Moscow State Circus,Rusia,1919,Sí,Tradicional,Uso continuo de fieras salvajes
6,Tihany Spectacular,Brasil,1945,No,Contemporáneo,Enfoque en ilusionismo y music-hall
7,New Shanghai Circus,China,1951,No,Tradicional,Enfoque en cultura milenaria
8,Circo de Chengdu,China,1953,No,Tradicional,Integración de ópera de Sichuan
9,Circus Vargas,EE.UU.,1969,No,Tradicional,Transición a shows 100% humanos


alt.LayerChart(...)

alt.HConcatChart(...)

,Nombre_Agrupación,Tipo_de_Circo,Región,Formación_Predominante,Estructura_Orgánica
0,Absurda Consecuencia,Contemporáneo,Araucanía,Talleres,Agrupación sin pers. jurídica
1,Balance,Contemporáneo,RM,Académica,Agrupación sin pers. jurídica
2,Casa Circleta,Contemporáneo,RM,Autodidacta,Agrupación sin pers. jurídica
3,Circo Ambulante,Contemporáneo,Valparaíso,Callejera,Personalidad jurídica
4,Circo Minero,Contemporáneo,RM,Universitaria,Personalidad jurídica
5,Circo Uach,Contemporáneo,Los Ríos,Universitaria,Agrupación sin pers. jurídica
6,Circo del Mundo,Contemporáneo,RM,Académica,Personalidad jurídica
7,Circonciente,Contemporáneo,RM,Social,Agrupación sin pers. jurídica
8,Cirkologia,Contemporáneo,RM,Académica,Personalidad jurídica
9,Diminuto Circus,Contemporáneo,RM,Académica,Agrupación sin pers. jurídica


alt.LayerChart(...)

alt.HConcatChart(...)